# Stage 1b: Embedding Re-ranking on OVOD Detections

For each detection produced by `OVOD_eval.ipynb`, crop the source frame at
the detection box, embed the crop with each candidate vision encoder, and
compute cosine similarity to the reference image embedding.

**Depends on `experiments/stage1/ovod_detections.jsonl`** — produced by the
OVOD notebook. Run OVOD first; this notebook fails fast if the file is
missing.

Outputs:

- `experiments/stage1/ssl_rerank.jsonl` — per-detection re-rank scores.
- `experiments/stage1/ssl_summary.json` — aggregated by (ssl_model,
  ovod_model) with category-split means, separation, top-1 accuracy, AUC.
- `experiments/stage1/ssl_heatmaps.png` — 4-panel heatmap
  (pos_easy / pos_hard / neg_easy / neg_hard) × (4 SSL × 8 OVOD).
- `experiments/stage1/ssl_distributions.png` — 4×8 grid of overlaid
  pos/neg histograms for visual distribution-overlap inspection.

Models:

- DINOv2 (`facebook/dinov2-base`, Apache 2.0)
- DINOv3 (`facebook/dinov3-vitb16-pretrain-lvd1689m`, restricted license — no military use)
- SigLIP 2 (`google/siglip2-base-patch16-256`, Apache 2.0)
- EVA-02 CLIP (loaded via `open_clip`)

In [ ]:
# Run once to install everything this notebook needs.
# Uses %pip so installs land in the kernel's environment, not the shell's.
# Re-running this cell is safe: pip skips packages already present.
%pip install --upgrade pip
%pip install --upgrade --quiet \
    torch torchvision pillow "numpy<2" \
    transformers accelerate \
    ultralytics \
    open_clip_torch

In [ ]:
import os

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image

from transformers import AutoImageProcessor, AutoModel, AutoProcessor

# open_clip is needed for EVA-02. Install with: uv add open_clip_torch
import open_clip

In [ ]:
REPO = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()).resolve()
OUTDIR = REPO / "experiments" / "stage1"
OUTDIR.mkdir(parents=True, exist_ok=True)

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
DTYPE = torch.float32

print("device:", DEVICE)


def free_memory():
    """Release tensors and clear the MPS cache between models."""
    gc.collect()
    if DEVICE == "mps":
        torch.mps.empty_cache()


# Per-detection re-rank rows; one per (ssl_model, detection)
rerank = []


def crop_box(img, box):
    """Crop img at box, clamped to image bounds. Returns None for empty boxes."""
    x0, y0, x1, y1 = (int(v) for v in box)
    x0 = max(0, x0)
    y0 = max(0, y0)
    x1 = min(img.width, x1)
    y1 = min(img.height, y1)
    if x1 <= x0 or y1 <= y0:
        return None
    return img.crop((x0, y0, x1, y1))


# Cache loaded frames so we don't re-decode the same PNG once per detection
frame_cache = {}


def get_frame(path):
    if path not in frame_cache:
        frame_cache[path] = Image.open(path).convert("RGB")
    return frame_cache[path]


# Load OVOD output
DETECTIONS_PATH = OUTDIR / "ovod_detections.jsonl"
if not DETECTIONS_PATH.exists():
    raise FileNotFoundError(
        f"{DETECTIONS_PATH} missing — run OVOD_eval.ipynb first to generate detection records"
    )

detections = [json.loads(line) for line in open(DETECTIONS_PATH)]
print("loaded", len(detections), "detections from", DETECTIONS_PATH)

reference = Image.open(REPO / "data" / "references" / "zbiotics.png").convert("RGB")


def run_ssl_model(ssl_model_name, embed_fn):
    """Embed reference once, then re-rank every detection with embed_fn.

    Appends one row per detection to the global `rerank` list. No per-frame
    image output is written — visualization is consolidated to heatmaps and
    distribution plots in the aggregate cell.
    """
    ref_emb = embed_fn(reference)
    n_done = 0
    for d in detections:
        img = get_frame(d["frame_path"])
        crop = crop_box(img, d["box"])
        if crop is None:
            continue
        t0 = time.perf_counter()
        e = embed_fn(crop)
        sim = float((e @ ref_emb.T).item())
        ms = (time.perf_counter() - t0) * 1000
        rerank.append({
            "ssl_model": ssl_model_name,
            "ovod_model": d["model"],
            "label": d["label"],
            "category": d["category"],
            "frame": d["frame"],
            "box": d["box"],
            "ovod_score": d["score"],
            "similarity": sim,
            "ms": ms,
        })
        n_done += 1
    print(ssl_model_name, "re-ranked", n_done, "detections")

In [ ]:
# DINOv2
model_id = "facebook/dinov2-base"
proc = AutoImageProcessor.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id, torch_dtype=DTYPE).to(DEVICE).eval()


def embed(img):
    """Return L2-normalized CLS-token embedding."""
    x = proc(images=img, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        feats = model(**x).last_hidden_state[:, 0]
    return F.normalize(feats, dim=-1)


run_ssl_model("dinov2_base", embed)

del model, proc
free_memory()

In [ ]:
# DINOv3 — restricted license (no military use)
model_id = "facebook/dinov3-vitb16-pretrain-lvd1689m"
proc = AutoImageProcessor.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id, torch_dtype=DTYPE).to(DEVICE).eval()


def embed(img):
    x = proc(images=img, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        feats = model(**x).last_hidden_state[:, 0]
    return F.normalize(feats, dim=-1)


run_ssl_model("dinov3_vitb16", embed)

del model, proc
free_memory()

In [ ]:
# SigLIP 2 — image tower only
model_id = "google/siglip2-base-patch16-256"
proc = AutoProcessor.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id, torch_dtype=DTYPE).to(DEVICE).eval()


def embed(img):
    x = proc(images=img, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.get_image_features(**x)
    # transformers 5.x returns BaseModelOutputWithPooling here, older versions returned a tensor
    feats = out.pooler_output if hasattr(out, "pooler_output") else out
    return F.normalize(feats, dim=-1)


run_ssl_model("siglip2_base", embed)

del model, proc
free_memory()

In [ ]:
# EVA-02 CLIP — loaded via open_clip
arch = "EVA02-B-16"
pretrained = "merged2b_s8b_b131k"

model, _, preprocess = open_clip.create_model_and_transforms(arch, pretrained=pretrained)
model = model.to(DEVICE).eval()


def embed(img):
    x = preprocess(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        feats = model.encode_image(x)
    return F.normalize(feats, dim=-1)


run_ssl_model("eva02_clip_b16", embed)

del model, preprocess
free_memory()

In [ ]:
# Aggregate: rerank.jsonl + summary.json + heatmaps.png + distributions.png
rerank_path = OUTDIR / "ssl_rerank.jsonl"
summary_json = OUTDIR / "ssl_summary.json"
heatmap_path = OUTDIR / "ssl_heatmaps.png"
distributions_path = OUTDIR / "ssl_distributions.png"

with open(rerank_path, "w") as f:
    for r in rerank:
        f.write(json.dumps(r) + "\n")


def auc(pos_scores, neg_scores):
    """Compute AUC = P(random positive > random negative). Returns NaN if either set empty."""
    if not pos_scores or not neg_scores:
        return float("nan")
    pos = np.array(pos_scores)
    neg = np.array(neg_scores)
    # Mann-Whitney U formulation
    n_pos, n_neg = len(pos), len(neg)
    diffs = pos[:, None] - neg[None, :]
    return float((np.sum(diffs > 0) + 0.5 * np.sum(diffs == 0)) / (n_pos * n_neg))


def top1_correctness(rows_for_pair):
    """Per-frame top-1 box: positive frames want their top-1 box to be from a positive frame.

    Defined as: for each positive frame, is the top-similarity detection in
    that frame ranked above ALL detections from that frame whose label is
    "neg"? Within a single frame all detections share the same label, so
    the meaningful version is: across all frames, is the per-frame top-1
    similarity higher on positive frames than on negative frames?
    """
    # Group by frame, take the top-1 similarity per frame, then ask
    # "what fraction of positive-frame top-1 are above the max of
    # negative-frame top-1?"
    by_frame = {}
    for r in rows_for_pair:
        key = (r["label"], r["category"], r["frame"])
        by_frame[key] = max(by_frame.get(key, -1.0), r["similarity"])
    pos_top1 = [v for (lab, _, _), v in by_frame.items() if lab == "pos"]
    neg_top1 = [v for (lab, _, _), v in by_frame.items() if lab == "neg"]
    if not pos_top1 or not neg_top1:
        return float("nan")
    threshold = max(neg_top1)
    return float(sum(1 for v in pos_top1 if v > threshold) / len(pos_top1))


# Aggregate by (ssl_model, ovod_model)
by_pair = {}
for r in rerank:
    key = (r["ssl_model"], r["ovod_model"])
    by_pair.setdefault(key, []).append(r)

summary_rows = []
for (ssl_model, ovod_model), rows in by_pair.items():
    cat_means = {}
    for bucket in ["pos_easy", "pos_hard", "neg_easy", "neg_hard"]:
        label, category = bucket.split("_")
        scores = [r["similarity"] for r in rows
                  if r["label"] == label and r["category"] == category]
        cat_means[bucket] = float(np.mean(scores)) if scores else None

    pos_scores = [r["similarity"] for r in rows if r["label"] == "pos"]
    neg_scores = [r["similarity"] for r in rows if r["label"] == "neg"]
    pos_mean = float(np.mean(pos_scores)) if pos_scores else None
    neg_mean = float(np.mean(neg_scores)) if neg_scores else None
    sep = (pos_mean - neg_mean) if (pos_mean is not None and neg_mean is not None) else None

    summary_rows.append({
        "ssl_model": ssl_model,
        "ovod_model": ovod_model,
        **cat_means,
        "pos_mean": pos_mean,
        "neg_mean": neg_mean,
        "separation": sep,
        "auc": auc(pos_scores, neg_scores),
        "top1_correctness": top1_correctness(rows),
        "n_pos": len(pos_scores),
        "n_neg": len(neg_scores),
    })

with open(summary_json, "w") as f:
    json.dump(summary_rows, f, indent=2)

print("wrote", rerank_path, "rows:", len(rerank))
print("wrote", summary_json)

In [ ]:
# 4-panel heatmap (pos_easy / pos_hard / neg_easy / neg_hard) × (SSL × OVOD)
ssl_models = sorted({row["ssl_model"] for row in summary_rows})
ovod_models = sorted({row["ovod_model"] for row in summary_rows})
buckets = [("pos_easy", "Greens", "pos_easy (higher = better)", False),
           ("pos_hard", "Greens", "pos_hard (higher = better)", False),
           ("neg_easy", "Greens_r", "neg_easy (lower = better)", True),
           ("neg_hard", "Greens_r", "neg_hard (lower = better)", True)]

fig, axes = plt.subplots(2, 2, figsize=(max(10, 1.2 * len(ovod_models) * 2), 1.2 * len(ssl_models) * 2 + 1))
axes = axes.flatten()
for ax, (bucket, cmap, title, _) in zip(axes, buckets):
    grid = np.full((len(ssl_models), len(ovod_models)), np.nan)
    for row in summary_rows:
        i = ssl_models.index(row["ssl_model"])
        j = ovod_models.index(row["ovod_model"])
        v = row.get(bucket)
        if v is not None:
            grid[i, j] = v
    im = ax.imshow(grid, aspect="auto", cmap=cmap, vmin=0.0, vmax=1.0)
    ax.set_xticks(range(len(ovod_models)))
    ax.set_xticklabels(ovod_models, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(ssl_models)))
    ax.set_yticklabels(ssl_models, fontsize=9)
    ax.set_title(title, fontsize=10)
    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            if not np.isnan(grid[i, j]):
                ax.text(j, i, f"{grid[i, j]:.2f}", ha="center", va="center",
                        color="black", fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)

fig.suptitle("SSL × OVOD: similarity by category (green = good)", fontsize=12)
fig.tight_layout()
fig.savefig(heatmap_path, dpi=120)
plt.close(fig)
print("wrote", heatmap_path)

In [ ]:
# Distribution overlap: 4 SSL rows × 8 OVOD cols, each panel is overlaid pos/neg histograms
fig, axes = plt.subplots(len(ssl_models), len(ovod_models),
                         figsize=(1.6 * len(ovod_models), 1.4 * len(ssl_models)),
                         sharex=True, sharey=True)
if len(ssl_models) == 1:
    axes = np.array([axes])
if len(ovod_models) == 1:
    axes = axes[:, None]

for i, ssl_model in enumerate(ssl_models):
    for j, ovod_model in enumerate(ovod_models):
        ax = axes[i, j]
        rows = [r for r in rerank if r["ssl_model"] == ssl_model and r["ovod_model"] == ovod_model]
        pos = [r["similarity"] for r in rows if r["label"] == "pos"]
        neg = [r["similarity"] for r in rows if r["label"] == "neg"]
        if pos:
            ax.hist(pos, bins=20, range=(0, 1), alpha=0.5, color="green", label="pos")
        if neg:
            ax.hist(neg, bins=20, range=(0, 1), alpha=0.5, color="red", label="neg")
        ax.set_xlim(0, 1)
        ax.tick_params(axis="both", labelsize=6)
        if i == 0:
            ax.set_title(ovod_model, fontsize=7, rotation=20)
        if j == 0:
            ax.set_ylabel(ssl_model, fontsize=8)

fig.suptitle("SSL × OVOD: similarity distributions (green=pos, red=neg)", fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(distributions_path, dpi=120)
plt.close(fig)
print("wrote", distributions_path)

# Print headline summary
print()
print("Top 5 (ssl_model, ovod_model) by separation:")
ranked = sorted(summary_rows, key=lambda r: r.get("separation") or -1, reverse=True)[:5]
for r in ranked:
    print(f"  {r['ssl_model']:18s} + {r['ovod_model']:22s}  sep={r['separation']:+.3f}  auc={r['auc']:.3f}  top1={r['top1_correctness']:.2f}")